#### Prompt Fine Tuning

In Prompt Tuning, we don't touch the layers; we modify the input stream.

We add a "Soft Prompt". It is a Learned Embedding Matrix that sits right at the start.

##### 1. Understanding Small Model for Fine Tuning

In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "false"
import torch
from transformers import pipeline

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"

In [47]:

model_pipeline = pipeline(
    "text-generation",
    model=model_id,
    dtype=torch.bfloat16,
    device_map="auto")

In [48]:
query_to_test="ServiceNow 'Notification' sent to a user but not received."

In [49]:
model_pipeline(query_to_test)[0]["generated_text"]

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"ServiceNow 'Notification' sent to a user but not received. What happens if the user doesn't click the app?\n\nThis is the code for 'Notification' from the app:\n\n```\nprivate void notification_add_message(sender, senderConfigurationEventArgs e) {\n    // Code to send the message\n    Message m = new Message(new String(e.MessageType.Message));\n    m.Send(Message.FINISHED);\n}\n\n```\n\nAs you can see, I am sending the message to the user without a reply to the message. How can I do the same in my code?username_1: I've found a solution to this problem.\n\n```\npublic class Message {\n\n    private final String message;\n\n    public Message(String message) {\n        this.message = message;\n    }\n\n    public String getMessage() {\n        return message;\n    }\n}\n\n```\n\nThis makes the code more readable.\nUpvotes: 1 username_2: You can use the `send` method on the `Message` class to send the message to the user.\n\nHere is the code:\n\n```\nprivate public void addMessage(sender

Here the response is not promising. Let's see how it response for general queries

In [50]:
model_pipeline("describe lion animal")[0]["generated_text"]

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'describe lion animal.\n"The lion is a very popular animal in the zoo. It\'s very easy to find in the zoo."\n"That\'s interesting. But you can\'t see the lion in the zoo."\n"But if you\'re interested in lions, you can find them in the zoo."\n"Yes, I can find the lion in the zoo. But I can\'t see it."\n"I can not find the lion in the zoo."\n"That\'s not true. I can find the lion in the zoo. It\'s a rare animal."\n"Yes, I can find the lion in the zoo."\n"But I cannot find it in the zoo."\n"It\'s a rare animal."\n"Yes, I can find the lion in the zoo."\n"But I cannot find it in the zoo."\n"The lion is a very popular animal in the zoo. It\'s very easy to find in the zoo."\n"That\'s interesting."\n"Yes, I can find the lion in the zoo. But I can not find it in the zoo."\n"I can not find the lion in the zoo."\n"That\'s not true. I can find the lion in the zoo."\n"It\'s a rare animal'

perfect, for general queries it is able to respond correctly. Now let's fine tune and see how it works for previous query

##### 2. Preparing dataset and loading it

In [51]:
from datasets import Dataset
import os

In [52]:
dataset = Dataset.from_text(os.path.join(os.getcwd(), "prompt_fine_tuning_training_data.jsonl"))

##### 3. Configuring Prompt Tuning

In [53]:
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model

In [54]:
config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    prompt_tuning_init=PromptTuningInit.TEXT, # as we are giving text, so embedding prompt initialization must be text only
    num_virtual_tokens=15,   # This will let llm to have virtual token embeddings for each input
    prompt_tuning_init_text="You are ServiceNow Assistant to provide resolution for IT related issues",  # giving persona to llm rather letting it set some default
    tokenizer_name_or_path=model_id # required in case init is Text
)

##### 4. Model Setup

In [55]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# AutoModelForCausalLM: To load Causal LM
# AutoTokenizer: Converts text into numerical format as per model system support

In [56]:
# loading base model
base_model = AutoModelForCausalLM.from_pretrained(model_id)


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4317.89it/s]


In [57]:
# loading it's tokenizer
base_model_tokenizer = AutoTokenizer.from_pretrained(model_id)

# for padding of the tokens - telling to llm that If you need to fill space to make the rows even, just use the 'End of Sentence' character
base_model_tokenizer.pad_token = base_model_tokenizer.eos_token

Tokenizing the training dataset

In [58]:
def tokenize_function(examples):
    return base_model_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128) 

In [59]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [60]:
tokenized_dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 160
})

In [61]:
# getting peft type model
peft_model = get_peft_model(base_model, config)

In [62]:
# let's see how many parameters will get trained
peft_model.print_trainable_parameters()

trainable params: 8,640 || all params: 134,523,648 || trainable%: 0.0064


##### 5. Training Arguments Setup

In [63]:
# import torch_directml
# device = torch_directml.device()
# peft_model.to(device)
 

In [64]:
training_args = TrainingArguments(
    output_dir="./servicenow-prompt-ft-training",
    learning_rate=2e-2, # for prompt ft, learning rate should be high, if we keep low it will not impact in the model as we are not making changes in weights
    num_train_epochs=1,
    per_device_eval_batch_size=10,
    save_strategy="no",
    logging_steps=1,
    fp16=True,                          # Crucial: Uses half-precision to save 50% VRAM
    optim="adamw_torch",                # Standard optimizer
)

##### 5. Let's start the training

In [65]:
# setting the Trainer configuration
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(base_model_tokenizer, mlm=False)
)

In [66]:
# triggering the training now
trainer.train()

Step,Training Loss
1,4.633420
2,3.991531
3,3.979479
4,3.856773
5,3.559228
6,3.484795
7,3.703253
8,3.579652
9,3.262565
10,3.598937


TrainOutput(global_step=20, training_loss=3.4795371294021606, metrics={'train_runtime': 5352.7261, 'train_samples_per_second': 0.03, 'train_steps_per_second': 0.004, 'total_flos': 13050280673280.0, 'train_loss': 3.4795371294021606, 'epoch': 1.0})

##### 6. Inference - Using Fine Tuned Model

peft library detects the finetuned model and calls that such that we do not need to merge fine tuned adapter with base mode for testing. However, if we want to deploy the model and use then we need to save and use it. Refer further sections

In [ ]:
from peft import PeftModel
def get_response_from_model(query: str, model: PeftModel, tokenizer: AutoTokenizer):
    # creating token for the user input in the same format as training data
    input_tokenizer = tokenizer(f"Issue: {query}. Resolution:", return_tensors="pt").to(model.device)

    # generating output 
    with torch.no_grad():
        output_tokens = model.generate(
            input_ids=input_tokenizer["input_ids"],
            max_new_tokens=100,
            temperature="0.5",
            pad_token_id=tokenizer.eos_token_id
        )
        full_text = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]
        resolution = full_text.split("Resolution:")[-1].strip()

        return resolution

In [ ]:
# let's generate the response
resolution = get_response_from_model(query_to_test, peft_model, base_model_tokenizer)
print(f"Resolution from Fine Tuned Model:\n{resolution}")

Resolution from Fine Tuned Model:
To resolve this issue, please contact the user directly. Thank you for your patience


Hmm, response is not up to the mark but now it is related to ServiceNow. When comparing with raw model this is acceptable.

let's save and see how we can load and use

##### 7. Saving fine tuned model

Fine-Tuned model is called 'adapter'

In [94]:
# saving model using trainer itself
trainer.save_model("./servicenow_prompt_ft_adapter")

##### 8. Load fine tuned model

To use fine tuned model (adapter) we have to merge with base model

In [95]:
adapter_path = "./servicenow_prompt_ft_adapter"

In [80]:
from peft import PeftModel, PeftConfig

In [96]:
ft_config = PeftConfig.from_pretrained(adapter_path)

# fethcing base model name fropm adapter config
base_model_name = ft_config.base_model_name_or_path

In [97]:
# loading base model and it's tokenizer
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

# loading tokenizer
base_tokenizer = AutoTokenizer.from_pretrained(adapter_path)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1083.20it/s]


In [98]:
# let's load the model 
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

# setting up the processing component
ft_model.to("cuda" if torch.cuda.is_available() else "cpu")

PeftModelForCausalLM(
  (base_model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(49152, 576)
      (layers): ModuleList(
        (0-29): 30 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=576, out_features=576, bias=False)
            (k_proj): Linear(in_features=576, out_features=192, bias=False)
            (v_proj): Linear(in_features=576, out_features=192, bias=False)
            (o_proj): Linear(in_features=576, out_features=576, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
            (up_proj): Linear(in_features=576, out_features=1536, bias=False)
            (down_proj): Linear(in_features=1536, out_features=576, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
          (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
     

let's run this

In [ ]:
# let's generate the response
resolution = get_response_from_model(query_to_test, ft_model, base_tokenizer)
print(f"Resolution from Fine Tuned Model:\n{resolution}")

d:\ProjectDesk\gh-iamatulkumar-ya\data-science\.venv_llm\Lib\site-packages\peft\peft_model.py:2219: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn(


Resolution from Fine Tuned Model:
To resolve this issue, please contact the user directly. Thank you for your patience


In [ ]:
# let's generate the response
resolution = get_response_from_model(query="Printer 'Default Gateway' is not reachable.", model=ft_model, tokenizer=base_tokenizer)
print(f"Resolution from Fine Tuned Model:\n{resolution}")

d:\ProjectDesk\gh-iamatulkumar-ya\data-science\.venv_llm\Lib\site-packages\peft\peft_model.py:2219: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn(


Resolution from Fine Tuned Model:
Update the printer's default gateway to the address you specified in the resolution.

**Step 5: Update the Printer**

* **Step 1:** Open the Control Panel.
* **Step 2:** Click on 'System and Security' > 'Update & Security' > 'Update Driver'.
* **Step 3:** Click on 'Update Driver' > 'Update Driver' > 'Update Driver'.
* **Step 4:** Click on 'Update Driver'
